# CineEmbed — EDA v2

Pipeline-first refresh of the EDA notebook. Applies all 13 fixes from the audit (see `docs/superpowers/specs/2026-05-03-eda-v2-design.md`). Produces a clean (329044, 451) feature matrix.

**Sections:**
- §1 Setup & Reproducibility
- §2 Pipeline Function Definitions
- §3 Pipeline Execution
- §4 EDA Visualizations
- §5 Persistence


In [ ]:
# §1 — Setup & Reproducibility
import os, json, hashlib, random, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from scipy import stats

# Optional GPU stack — only imported if available
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CONFIG = {
    'seed': 42,
    'data_dir': Path('data'),
    'artifacts_dir': Path('artifacts'),
    'figures_dir': Path('artifacts/figures'),

    # File paths
    'paths': {
        'details': Path('data/AllMoviesDetailsCleaned.csv'),
        'casting': Path('data/AllMoviesCastingRaw.csv'),
        'awards':  Path('data/220k_awards_by_directors.csv'),
    },

    # Feature engineering knobs (single source of truth — fix #11 + clean ablation)
    'top_n_genres': 20,
    'top_n_languages': 30,
    'q99_clip_threshold': 0.99,
    'runtime_clip': (10, 300),

    # Embedding model (fix #5 — multilingual)
    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',
    'embedding_batch_size': 64,
    'embedding_dim': 384,
    'embedding_cache': Path('artifacts/text_embeddings.npy'),
    'embedding_meta': Path('artifacts/text_embeddings.meta.json'),
}

CONFIG['artifacts_dir'].mkdir(parents=True, exist_ok=True)
CONFIG['figures_dir'].mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG['seed'])

# Reproducibility self-check
_check = np.random.rand(3)
print("\u2705 \u00a71 Setup complete")
print(f"   seed = {CONFIG['seed']}")
print(f"   np.random sample (deterministic) = {_check}")
print(f"   torch available = {HAS_TORCH}")
